In [1]:
!pip install langchain[openai] google-search-results serpapi

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py) ... - \ | done
  DEPRECATION: Building 'google-search-results' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'google-search-results'. Discussion can be found at https://github.com/pypa/pip/issues/6334

  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32003 sha256=d25ebb4754c4f43cdef6ecb16f5f1feffa4617b419037b5806f8d65852f0fa8b
  Stored in directory: /voc/work/.cache/pip/wheels/d3/b2/c3/03302d12bb44a2cdff3c9371f31b72c0c4e84b8d2285eeac53
Successfully built google-search-results

   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [

In [2]:
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import requests
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Core imports for LLM and agents
from langchain.tools import Tool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, load_tools
# PromptTemplate for constructing the final prompt string
from langchain.prompts import PromptTemplate

/usr/local/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [4]:
# 1. Initialize the LLM with deterministic behavior (temperature=0)
llm = ChatOpenAI(temperature=0.2, model="gpt-3.5-turbo")

In [5]:
# 2. Load tools:
#    - "serpapi": web search
#    - "llm-math": math through the LLM
serptool = load_tools(
    ["serpapi"],
    llm=llm
)

mathtool = load_tools(
    ["llm-math"],
    llm = llm
)

combinetool = load_tools(
    ["serpapi", "llm-math"],
    llm = llm
)

In [6]:
# 3. Create an agent that uses ReAct-style reasoning with descriptions
serpagent = initialize_agent(
    tools=serptool,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=False
)

mathagent = initialize_agent(
    tools=mathtool,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=False
)

combineagent = initialize_agent(
    tools=combinetool,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=False
)

/tmp/ipykernel_33/1131171363.py:2: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  serpagent = initialize_agent(


In [7]:
# Here we define a PromptTemplate that:
# - Explains to the LLM that it has access to a search tool and a calculator.
# - Instructs it to first use search, then do math with the result.
# - Inserts the user question via {question}.

prompt_template_Search = PromptTemplate(
    input_variables=["question"],
    template="""
You are an AI assistant that can use tools to answer questions.

You have access to:
1. A web search tool (serpapi) to look up real-world facts.

For the given task, follow these steps:
- First, use the web search tool to find any factual information you need.
- Finally, explain your reasoning and provide a clear final answer.

Now, here is the task you must solve:
{question}
""".strip()
)



prompt_template_Math = PromptTemplate(
    input_variables=["question"],
    template="""
You are an AI assistant that can use tools to answer questions.

You have access to:
1. A math tool (llm-math) to perform precise numeric calculations.

For the given task, follow these steps:
- Then, use the math tool to perform any required calculations.
- Finally, explain your reasoning and provide a clear final answer.

Now, here is the task you must solve:
{question}
""".strip()
)


prompt_template_Combine = PromptTemplate(
    input_variables=["question"],
    template="""
You are an AI assistant that can use tools to answer questions.

You have access to:
1. A web search tool (serpapi) to look up real-world facts.
2. A math tool (llm-math) to perform precise numeric calculations.

For the given task, follow these steps:
- First, use the web search tool to find any factual information you need.
- Then, use the math tool to perform any required calculations.
- Finally, explain your reasoning and provide a clear final answer.

Now, here is the task you must solve:
{question}
""".strip()
)

In [8]:
# Define a realistic, simple question that needs BOTH tools:
# - Use search to find Google's founding year.
# - Use math to add 25 to that year.
search_question_text = """
Who is the current CEO of Microsoft?
""".strip()

math_question_text = """
What is 512/16?
""".strip()

combine_question_text = """
What is the square root of 81 plus the current US President’s first name?
""".strip()

In [10]:
# Use the PromptTemplate to create the final prompt string
search_formatted_prompt = prompt_template_Search.format(question=search_question_text)

# Pass the formatted prompt STRING to the agent
search_result = serpagent.run(search_formatted_prompt)

# Print the final answer from the agent
print("\n===== FINAL ANSWER =====\n")
print(search_result)

/tmp/ipykernel_33/1568944646.py:5: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  search_result = serpagent.run(search_formatted_prompt)

===== FINAL ANSWER =====

Satya Nadella


In [11]:
# Use the PromptTemplate to create the final prompt string
math_formatted_prompt = prompt_template_Math.format(question=math_question_text)

# Pass the formatted prompt STRING to the agent
math_result = mathagent.run(math_formatted_prompt)

# Print the final answer from the agent
print("\n===== FINAL ANSWER =====\n")
print(math_result)


===== FINAL ANSWER =====

32.0


In [12]:
# Use the PromptTemplate to create the final prompt string
combine_formatted_prompt = prompt_template_Combine.format(question=combine_question_text)

# Pass the formatted prompt STRING to the agent
combine_result = combineagent.run(combine_formatted_prompt)

# Print the final answer from the agent
print("\n===== FINAL ANSWER =====\n")
print(combine_result)


===== FINAL ANSWER =====

13.0
